In [24]:
import os
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader, PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [25]:
# 1. Setup Data 
# Create a folder named 'data' and put your 'testingdata.txt' and any PDFs there!
# This loader looks into a 'data' folder for any .txt or .pdf files
txt_loader = DirectoryLoader("./RAG_data", glob="./*.txt", loader_cls=TextLoader)
pdf_loader = DirectoryLoader("./RAG_data", glob="./*.pdf", loader_cls=PyPDFLoader)

# Load everything
docs = txt_loader.load() + pdf_loader.load()

# 'Recursive' splitting is the 2026 standard—it respects double newlines, then single, then spaces
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=200,
    add_start_index=True # Helpful for citations later
)
splits = text_splitter.split_documents(docs)

print(f"Loaded {len(docs)} documents and created {len(splits)} chunks.")

Loaded 73 documents and created 302 chunks.


In [26]:
# 2. Vector Store
# This downloads a small, high-performance model to your machine
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create the database in-memory (or add persist_directory="db" to save it)
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("Vector database is ready.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector database is ready.


In [29]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter

llm = ChatOllama(model="llama3.2", temperature=0)

# 1. The prompt that tells the AI to use context and cite sources
template = """You are a helpful assistant. Answer the question using ONLY the context provided.
If the answer isn't in the context, say you don't know.

Context:
{context}

Question: {question}

Answer (and cite the filename used):"""

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that cites sources."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", template),
])

# 2. Function to format docs with metadata (Source tracking)
def format_docs(docs):
    return "\n\n".join(f"Source: {d.metadata.get('source', 'Unknown')}\nContent: {d.page_content}" for d in docs)

# 3. The Modern LCEL RAG chain (Fixed mapping)
rag_chain = (
    {
        # We use itemgetter to pull just the string question for the retriever
        "context": itemgetter("question") | retriever | format_docs,
        "question": itemgetter("question"),
        "chat_history": itemgetter("chat_history")
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history = [] # This stores your conversation

def ask_question(query):
    global chat_history
    
    # Run the chain
    result = rag_chain.invoke({
        "question": query,
        "chat_history": chat_history
    })
    
    # Update history
    chat_history.append(HumanMessage(content=query))
    chat_history.append(AIMessage(content=result))
    
    return result



Question 1: What are the energy transfer strategies?
AI: According to the provided context, there are two main energy transfer strategies:

1. **Decentralized Strategy**: This strategy is not explicitly described in the given content, but it can be inferred that it involves nodes measuring their energy levels, classifying themselves into appropriate energy status, and independently deciding whether to harvest energy based on a decentralized algorithm.

2. **Centralized Energy Transfer Strategy**: This strategy involves a centralized decision-making process where:
	* The OBN (Optimization Body Network) periodically reports its energy level to the network.
	* The OBN's functionality is split into two key components: 
		+ A predictor estimates the future energy levels of each node and sends these estimates to the decision-maker.
		+ The decision-maker selects a set of nodes for energy replenishment based on these predictions, aiming to maximize energy efficiency across the network.

(Cent

In [35]:
# --- TEST IT ---
print("Question 1: What are the energy transfer strategies?")
print(f"AI: {ask_question('What are the energy transfer strategies?')}\n")

print("Question 2: What is the novelty in using NARS scheduler?")
print(f"AI: {ask_question('What is the novelty in using NARS scheduler?')}\n")

print("Question 3: What did they mean by SINDy+RL?") 
print(f"AI: {ask_question('What did they mean by SINDy+RL?')}\n")

print("Question 4: How did they reduce the number of communications?") 
print(f"AI: {ask_question('How did they reduce the number of communications?')}\n")

print("Question 5: How did they reduce the number of communications in the energy-accuracy paper?") 
print(f"AI: {ask_question('How did they reduce the number of communications in the energy-accuracy paper?')}\n")

Question 1: What are the energy transfer strategies?
AI: According to the context provided, there are two main energy transfer strategies:

1. **Decentralized Strategy**: This strategy is not explicitly described in detail, but it involves nodes measuring their energy levels, classifying themselves into appropriate energy status, and independently deciding whether to harvest energy based on a decentralized algorithm.
2. **Centralized Energy Transfer Strategy**: This strategy involves:
	* The OBN (Optimization Body Network) periodically reporting its energy level to the network.
	* The OBN's functionality is split into two key components: 
		+ A predictor estimates the future energy levels of each node and sends these estimates to the decision-maker.
		+ The decision-maker selects a set of nodes for energy replenishment based on these predictions, aiming to maximize energy efficiency across the network.

(Cited from RAG_data/Energy_Transfer_Strategies_in_Magnetic_Resonance_Based_Intrabo

# Technical Report: Local Conversational RAG System
Project Overview
This system implements a Retrieval-Augmented Generation (RAG) pipeline using a local-first architecture. It allows users to query a private repository of documents (PDFs and Text files) while maintaining a natural, multi-turn conversation history without sending sensitive data to external APIs.

## 1. System Architecture

The pipeline is built on the ETL (Extract, Transform, Load) framework for data and LCEL (LangChain Expression Language) for real-time inference.

Component	Technology	Role
Loader	DirectoryLoader	Aggregates disparate file formats (PDF, TXT) into a unified stream.
Splitter	RecursiveCharacterTextSplitter	Breaks text into 1000-character chunks while preserving semantic headers.
Embeddings	HuggingFace (all-MiniLM-L6-v2)	Converts text into numerical vectors representing semantic meaning.
Vector Store	ChromaDB	A high-speed local database for similarity searching.
LLM	Ollama (Llama 3.2)	Generates human-like text based on the retrieved context.
## 2. Implementation Details

### A. Multi-Format Data Ingestion

Unlike basic systems that only read strings, this system uses a DirectoryLoader configured with PyPDFLoader. This allows the system to parse:

Unstructured Text: Standard .txt files.

Digital PDFs: Searchable documents containing tables and formatted text.

### B. Semantic Chunking

We use a Recursive Character Text Splitter. Instead of cutting text at arbitrary character counts, it attempts to split by paragraphs, then sentences, and finally words. This ensures that the context passed to the AI is logically complete and coherent.

### C. Conversational Memory

Standard RAG systems are "stateless." We implemented a Chat History Buffer using MessagesPlaceholder.

The Benefit: It enables the use of pronouns. For example, if you ask "Who is the lead?" after a previous question about "Project Albatross," the system understands the context of "it."

### D. Grounding & Citations

To ensure accuracy and prevent AI "hallucinations," the system is strictly grounded via the system prompt. Every response is required to cite its source from the metadata (the filename), providing a clear audit trail for the user.

## 3. The Logic Flow

User Query: The user enters a natural language question.

Retrieval: The system converts the query into a vector and finds the top 3 most relevant chunks in ChromaDB.

Context Injection: The system retrieves the text chunks and the previous chat history.

Synthesis: The LLM (Llama 3.2) generates an answer based only on the retrieved data.

Output: The final answer is displayed with the source filename attached.